# 03 · 거대 모델로 그냥 한 번 — SAM2 zero-shot

> **공부 기록 노트북 3번.** 01·02는 *우리가 직접 만든 베이스라인*이었습니다.
> 이번엔 이미 세상에 나와있는 **거대 분할 모델 SAM2**를 가져다가
> *학습 없이* 우리 데이터에 그냥 써봅니다.

## 01·02 복습

- 01: CholecSeg8k 데이터를 이해했다 (6-class, 영상 단위 split, 불균형).
- 02: EfficientNet-B4 U-Net 베이스라인을 미니 학습했다. loss → 줄어듦.

## 이 노트북에서 할 일

1. **foundation model** 과 **zero-shot** 이 무슨 뜻인지 이해
2. **SAM2** 모델을 HuggingFace에서 받아 GPU에 올린다
3. **수술 프레임의 쓸개를 한 번 클릭**해서 SAM2가 뭘 분할하는지 본다
4. SAM2 의 **3개 후보 마스크(multimask)** 가 의미하는 것 이해
5. 여러 프레임에서 **IoU 숫자**로 측정 — "프롬프트가 있을 때 SAM2 는 얼마나 잘 하는가"
6. zero-shot 의 **한계**: 다음 노트북(04) 에서 LoRA 미세조정으로 메울 것

⚠️ **GPU 런타임 필요** (T4 정도면 충분 · SAM2 base-plus ~80M 파라미터).

## 1. Foundation model / zero-shot / SAM2

### Foundation model 이 뭔가요?

비유로 설명하면:

> **전문가**는 하나만 잘 합니다 (02 의 U-Net 은 6 개 해부구조만 알아요).
> **만물박사**는 거의 모든 걸 어느 정도 합니다 — 사람으로 치면
> "온갖 책을 읽은 박학다식한 사람".

**Foundation model** 은 그 만물박사입니다. 인터넷에서 모은 *수백만 ~ 수십억
장*의 이미지로 미리 학습된 거대 모델로, 한 가지 일을 위해 만들어진 게 아니라
*나중에 어떤 일에든 갖다 쓸 수 있게* 일반적인 시각 지식을 쌓아둔 모델입니다.

### Zero-shot 이 뭔가요?

> **Zero-shot** = "**우리 데이터로 한 번도 학습 안 시키고** 그냥 결과만 보기."

말 그대로 0 번의 추가 학습. 02 에서 미니 학습으로 loss 가 줄어드는 걸 봤죠?
zero-shot 은 그 단계가 *아예 없습니다*. 모델 받아서, 그대로 추론만.

### SAM2 (Segment Anything Model 2)

Meta 가 만든 **분할 전용** foundation model 입니다. 일반 사진·동영상의
10 억+ 마스크로 학습되어 *물체의 경계*에 대한 일반적인 감각을 가지고 있어요.

특징 3 가지:

1. **Class-agnostic** — "이건 쓸개"라는 의미는 모릅니다. *물체의 경계*만
   압니다. 클릭한 자리에 *뭐가 있든* 그 윤곽을 따라 칠해 줍니다.
2. **Promptable** — 사용자가 "여기"라고 알려줘야 합니다. 그 신호는
   **점(click)** 이나 **박스**입니다.
3. **Multimask** — 클릭 하나로는 의도가 종종 애매해서 (셔츠? 사람 전체?
   특정 부위?) **3 가지 후보 마스크**를 줍니다.

```
        [frame]                                [SAM2's mask]
         . . .                                    ....
       . * .         ──(여기 클릭)──►          ..##..
         . .                                    ....
```

> 키워드: foundation model, zero-shot, SAM (Segment Anything), promptable
> segmentation, multimask output

## 0. 환경 준비

01·02 와 동일한 패턴 — repo 가져오고 의존성 설치. 이미 돼있으면 새로 받지 않고
최신 코드만 당겨옵니다. **GPU 필요해요**.

In [ ]:
%cd /content
import os
if not os.path.isdir("surgical-ai"):
    !git clone -b main https://github.com/duck-bin/surgical-ai.git
%cd surgical-ai
!git pull --ff-only
!bash scripts/colab_setup.sh

import torch
print("준비 완료 ·", os.getcwd())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "없음 → Runtime > Change runtime type > GPU 로 바꾸세요")

### (선택) Google Drive 연결 — 다운로드·체크포인트 보존

Colab은 런타임이 초기화되면 `/content`가 비워져 데이터(3GB+)와 학습 체크포인트가
사라집니다. 아래 셀은 Drive를 마운트해 `data/`·`outputs/`·모델 캐시를 Drive에
저장하므로, **다음에 다시 와도 받아둔 데이터와 학습 결과를 그대로** 씁니다.

**기본값은 `False`(Drive 미사용)** — 전부 `/content` 에서 돌아갑니다. Drive 에 보존하고 싶을 때만 `USE_DRIVE = True` 로 바꾸세요(인증 창이 뜹니다).

In [ ]:
USE_DRIVE = False  # Drive 에 데이터·체크포인트를 보존하려면 True (Drive 없으면 그대로 두세요)
if USE_DRIVE:
    from src.utils.colab import mount_drive
    mount_drive()

## 2. SAM2 모델 받기

HuggingFace Hub 에서 `facebook/sam2-hiera-base-plus` 체크포인트(~330MB) 를
받아옵니다. 첫 실행에서만 다운로드, 그 다음부턴 캐시에서 바로 읽어요.

- `Sam2Model` — 신경망 본체 (Hiera 이미지 인코더 + prompt 인코더 + mask 디코더)
- `Sam2Processor` — 입력 전처리 + 마스크 후처리를 도와주는 헬퍼

> 키워드: HuggingFace transformers, Sam2Model, Sam2Processor, model checkpoint

In [ ]:
import torch
from transformers import Sam2Model, Sam2Processor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

CHECKPOINT = "facebook/sam2-hiera-base-plus"
sam2_model = Sam2Model.from_pretrained(CHECKPOINT).to(device).eval()
sam2_processor = Sam2Processor.from_pretrained(CHECKPOINT)

n_params = sum(p.numel() for p in sam2_model.parameters())
print(f"loaded {CHECKPOINT}: {n_params / 1e6:.0f}M params")

## 3. 한 번 클릭해 보기

이제 SAM2 를 써봅니다. 실험은 이렇습니다:

1. CholecSeg8k val 세트에서 **쓸개가 잘 보이는** 프레임을 하나 고른다
2. 정답 마스크에서 **쓸개 영역의 한 점**을 골라 SAM2 에 "여기" 라고 알려줌
3. SAM2 가 칠한 마스크를 ground truth (정답) 와 비교

핵심: SAM2 는 우리가 "쓸개"라고 *알려준 적 없습니다*. 그냥 그 점의 물체 윤곽
을 따라갈 뿐이에요. 그래도 얼마나 그럴듯한지 보는 게 목표.

### IoU 한 줄 정의

```
IoU = (예측과 정답이 겹치는 픽셀 수) / (둘 중 하나라도 칠해진 픽셀 수)
```

1.0 이면 완벽 일치, 0 이면 전혀 겹치지 않음.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.data.cholecseg8k import CholecSeg8kDataset, CLASS_NAMES

# val 세트에서 쓸개가 잘 보이는 프레임 하나 찾기
val_ds = CholecSeg8kDataset(split="val", image_size=512, transform=None)
TARGET_CLASS = CLASS_NAMES.index("gallbladder")   # = 2
sample = None
for i in range(len(val_ds)):
    candidate = val_ds[i]
    if (candidate["mask"].numpy() == TARGET_CLASS).sum() > 1000:
        sample = candidate
        print(f"using val frame #{i}")
        break

image = (sample["image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
gt = sample["mask"].numpy()
gt_region = (gt == TARGET_CLASS)   # (H, W) bool — 진짜 쓸개 영역

# 쓸개 영역 안에서 점 하나 — 중앙쯤
ys, xs = np.where(gt_region)
mid = len(xs) // 2
point = [int(xs[mid]), int(ys[mid])]
print(f"click at (x, y) = {point}")

# SAM2 에게 "여기" 라고 알려주기
inputs = sam2_processor(
    images=image,
    input_points=[[[point]]],   # [batch][object][point][x, y]
    input_labels=[[[1]]],       # 1 = foreground point
    return_tensors="pt",
).to(device)
with torch.no_grad():
    outputs = sam2_model(**inputs, multimask_output=True)

# 후처리 — 원본 해상도의 binary mask 로 변환
masks = sam2_processor.post_process_masks(
    outputs.pred_masks, inputs["original_sizes"]
)[0]
if masks.ndim == 4:           # (num_objects, num_masks, H, W) -> drop num_objects
    masks = masks[0]
# masks: (num_masks=3, H, W) bool

# SAM2 가 가장 확신하는 마스크 하나 고르기
best_idx = int(outputs.iou_scores[0, 0].argmax())
best_mask = masks[best_idx].cpu().numpy()

inter = (best_mask & gt_region).sum()
union = (best_mask | gt_region).sum()
iou_val = inter / union if union else 0.0
print(f"best candidate IoU vs ground truth: {iou_val:.3f}")

# 시각화
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(image)
ax[0].scatter([point[0]], [point[1]], c="red", s=300, marker="*",
              edgecolor="white", linewidth=2)
ax[0].set_title("frame + click")

sam_overlay = image.copy()
sam_overlay[best_mask] = (sam_overlay[best_mask] * 0.5
                          + np.array([255, 60, 60]) * 0.5).astype(np.uint8)
ax[1].imshow(sam_overlay)
ax[1].set_title(f"SAM2 best mask (IoU = {iou_val:.2f})")

gt_overlay = image.copy()
gt_overlay[gt_region] = (gt_overlay[gt_region] * 0.5
                         + np.array([60, 255, 60]) * 0.5).astype(np.uint8)
ax[2].imshow(gt_overlay)
ax[2].set_title("ground truth (gallbladder)")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 4. SAM2 는 마스크를 3 개 줍니다 — 왜?

위에서 우리는 "best 하나"만 봤어요. 실제로 SAM2 는 **3 개 후보**를 줍니다.

왜 3 개? 점 하나만으론 의도가 종종 *애매*합니다. 사진 속 사람 옷을 클릭했을
때:
- 옷만? (가장 작은 단위)
- 옷의 한 부분만? (중간 단위)
- 사람 전체? (큰 단위)

세 가지 다 "맞을 수도" 있습니다. SAM2 는 그래서 다양한 크기·해석의 마스크
3 개를 같이 내놓고, *각각에 대한 자기 확신 점수(SAM2 score)* 도 줍니다.

아래는 위와 같은 클릭에 대한 **3 개 후보 전부**를 그려본 것 — 어느 것이
우리가 원하는 쓸개에 가장 가까운지 직접 비교해 보세요.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 5))
ax[0].imshow(image)
ax[0].scatter([point[0]], [point[1]], c="red", s=300, marker="*",
              edgecolor="white", linewidth=2)
ax[0].set_title("click")
for i in range(3):
    candidate = masks[i].cpu().numpy()
    score = float(outputs.iou_scores[0, 0, i])
    overlay = image.copy()
    overlay[candidate] = (overlay[candidate] * 0.5
                          + np.array([255, 60, 60]) * 0.5).astype(np.uint8)
    ax[i + 1].imshow(overlay)
    ax[i + 1].set_title(f"candidate {i} (SAM2 score: {score:.2f})")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 5. 숫자로 측정 — 클래스별 "프롬프트 상한선"

그림 하나만으로는 SAM2 가 얼마나 잘하는지 모릅니다. **숫자**가 필요해요.

측정 절차:
1. val 세트에서 여러 프레임을 뽑는다
2. 각 프레임에서 *foreground class 중 보이는 것* 마다 (배경 제외)
3. 정답 영역 안에서 **점 하나를 무작위로** 뽑아 SAM2 에 알려줌
4. SAM2 가 준 3 개 중 *정답에 가장 가까운* 마스크의 IoU 를 기록
5. 클래스별 평균 IoU 출력

이 숫자가 의미하는 것: **"우리가 SAM2 에 완벽한 단서(정답 영역 안의 점)를
주면, SAM2 가 zero-shot 으로 어디까지 할 수 있는가."** 일종의 *상한선
(upper bound)* 입니다 — 실제 시스템은 자동으로 동작해야 하므로 이만큼은
나오기 어려워요.

머릿속으로 02 의 U-Net 미니 학습 결과(160 장·3 epoch) 와 비교해 보세요.

In [ ]:
def sam2_best_mask(img, pt):
    """Prompt SAM2 with one point; return its best (highest-score) mask."""
    inputs = sam2_processor(
        images=img,
        input_points=[[[pt]]],
        input_labels=[[[1]]],
        return_tensors="pt",
    ).to(device)
    with torch.no_grad():
        outputs = sam2_model(**inputs, multimask_output=True)
    masks = sam2_processor.post_process_masks(
        outputs.pred_masks, inputs["original_sizes"]
    )[0]
    if masks.ndim == 4:
        masks = masks[0]
    best = int(outputs.iou_scores[0, 0].argmax())
    return masks[best].cpu().numpy().astype(bool)


def compute_iou(pred, target):
    inter = (pred & target).sum()
    union = (pred | target).sum()
    return float(inter) / float(union) if union else float("nan")


# 평가 루프 — N 프레임에 대해 각 foreground class 당 한 점씩 prompt
N_FRAMES = 20
MIN_PX = 200
rng = np.random.default_rng(42)
per_class = {c: [] for c in range(1, len(CLASS_NAMES))}   # 1..5 (foreground only)

for i in range(min(N_FRAMES, len(val_ds))):
    s = val_ds[i]
    img_np = (s["image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    gt_i = s["mask"].numpy()
    for c in range(1, len(CLASS_NAMES)):
        region = (gt_i == c)
        if region.sum() < MIN_PX:
            continue
        ys_c, xs_c = np.where(region)
        idx = int(rng.integers(len(xs_c)))
        pred = sam2_best_mask(img_np, [int(xs_c[idx]), int(ys_c[idx])])
        per_class[c].append(compute_iou(pred, region))

print(f"\n{'class':16s} {'mean IoU':>10s} {'samples':>10s}")
for c, scores in per_class.items():
    if scores:
        print(f"{CLASS_NAMES[c]:16s} {np.mean(scores):10.3f} {len(scores):10d}")
    else:
        print(f"{CLASS_NAMES[c]:16s} {'-':>10s} {'0':>10s}")

## 6. zero-shot 의 한계 — 다음 노트북에서 채울 것

위 숫자를 보면 SAM2 는 *프롬프트가 있으면* 꽤 잘 합니다. 그런데 실제 시스템은
이렇게 못 써요:

1. **자동 동작이 안 됨** — 매 프레임마다 사람이 클릭할 수는 없습니다. 시스템
   은 *입력만 받아서* 6-class 마스크를 *자동으로* 내놔야 합니다.
2. **해부 지식이 없음** — SAM2 는 cystic duct 와 cystic artery 가 비슷한
   *얇은 관*이라는 걸 알 뿐, 둘을 구분 못 합니다. 클릭한 자리에 따라 둘 중
   하나를 칠하는데, 우리는 *어느 것인지*를 답해야 해요.

→ **노트북 04 (`04_sam2_lora.ipynb`)** 에서:
   - SAM2 인코더는 그대로 두고 (그 일반 지식이 가치 있음)
   - **LoRA** 라는 작은 부품만 추가로 학습시켜서
   - 우리 6 개 클래스를 *프롬프트 없이* 직접 출력하게 만든다

그게 *이 프로젝트의 주력 모델*입니다.

## 마무리 — 이 노트북 정리

### 이 노트북에서 배운 것

- **Foundation model** = 거대한 일반 지식을 가진 모델 (만물박사).
- **Zero-shot** = 우리 데이터로 추가 학습 없이 그대로 사용.
- **SAM2** = 분할 전용 foundation model. 특징:
  - class-agnostic (의미 모름, 경계만 압)
  - promptable (점/박스로 "여기" 알려줘야 함)
  - multimask (애매성 때문에 3 개 후보를 줌)
- 점 하나로 프롬프트해서 best-of-3 IoU 를 재면 **상한선** 으로서 의미 있음.
- 그런데 자동 동작이 안 되고 해부 지식이 없음 → 실제 시스템엔 부족.

### 아직 이해가 덜 된 것 / 더 파볼 것

- SAM2 내부는 어떻게 생겼나? (Hiera 인코더 + mask 디코더) → `src/models/sam2_lora.py`
  에서 우리가 어떻게 만지는지 다음 노트북에서 본다.
- 클릭이 아니라 **박스** 프롬프트를 주면? (`input_boxes` 인자) — 시간 되면 시도.
- multimask 의 SAM2 자체 score 와 *진짜* IoU 가 일치할까? (모델의 자기 평가
  정확도)

### 다음 노트북

**`04_sam2_lora.ipynb`** — SAM2 를 우리 데이터에 맞게 **LoRA 미세조정**.
프롬프트 없이도 6-class 마스크를 직접 내놓도록 만든다. 인코더는 거의 freeze,
*작은 어댑터*만 학습시켜 효율적으로.